# 🧪 معمل التجارب: بناء مسار RAG كامل مع التقطيع الدلالي المعياري (Lecture 39)
أهلاً بك في معملك التطبيقي للمحاضرة 39 من مرحلة التقطيع الدلالي المتقدم (`Advanced Semantic Chunking`)! 🚀

**فكرة المعمل ببساطة:**
في الدروس السابقة كتبنا كود التقطيع الدلالي كدوال متفرقة.
اليوم سننظم شغلنا باحترافية كمهندسي إنتاج (`Production AI Engineers`):
1. هنبني كلاس معياري منظم `ThresholdSemanticChunker` يقبل وثائق `Document` من LangChain ويحافظ على بياناتها الوصفية (`Metadata`).
2. هنخزن القطع الناتجة في مخزن متجهات `FAISS`.
3. هنبني مسار RAG كامل باستخدام `LCEL` و `Groq LLM` فائق السرعة، ونرى كيف يتم استرجاع الإجابات بنقاء دلالي 100% دون أي تشتت!

> 💡 **طريقة التدريب في المعمل:**
> اقرأ الشرح المبسط ⬅️ شوف الكود المرجعي الجاهز وشغله ⬅️ ثم طبق بنفسك في الخلية الفارغة المخصصة لك تحت كل خطوة!

### 1. استيراد المكتبات وتهيئة البيئة
**الفكرة ببساطة:**
بنستورد أدوات LangChain Core الأساسية، وموديل التضمين المحلي، ومخزن `FAISS`، ونموذج `Groq` اللغوي السريع.

In [ ]:
# 1. import core libraries and load environment keys
import os
import warnings
from dotenv import load_dotenv
from typing import List

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

warnings.filterwarnings("ignore")
load_dotenv()

print("✅ Libraries imported successfully!")
print(f"Groq API Key available: {bool(os.getenv('GROQ_API_KEY'))}")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب استيراد المكتبات وفحص المتغيرات هنا وشغلها):


### 2. بناء كلاس التقطيع المعياري (`ThresholdSemanticChunker`)
**الفكرة ببساطة:**
الكلاس يحتوي على:
- `__init__`: لتحديد موديل التضمين والعتبة (افتراضياً 0.7).
- `split_text`: تأخذ نصاً مجرداً وتقسمه لجمل وتجمعه دلالياً.
- `split_documents`: تأخذ كائنات `Document` وتخرج كائنات `Document` مقطعة مع الحفاظ على الـ `metadata`.

In [ ]:
# modular semantic chunker class
class ThresholdSemanticChunker:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2", threshold: float = 0.7):
        self.model = SentenceTransformer(model_name)
        self.threshold = threshold

    def split_text(self, text: str) -> List[str]:
        sentences = [s.strip() for s in text.split("\n") if s.strip()]
        if not sentences:
            return []
        
        embeddings = self.model.encode(sentences)
        chunks = []
        current_chunk = [sentences[0]]

        for i in range(1, len(sentences)):
            sim = cosine_similarity([embeddings[i - 1]], [embeddings[i]])[0][0]
            if sim >= self.threshold:
                current_chunk.append(sentences[i])
            else:
                chunks.append(" ".join(current_chunk))
                current_chunk = [sentences[i]]

        if current_chunk:
            chunks.append(" ".join(current_chunk))
        return chunks

    def split_documents(self, docs: List[Document]) -> List[Document]:
        result = []
        for doc in docs:
            chunks = self.split_text(doc.page_content)
            for chunk in chunks:
                result.append(
                    Document(
                        page_content=chunk,
                        metadata=doc.metadata.copy()
                    )
                )
        return result

print("✅ ThresholdSemanticChunker class defined successfully!")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب تعريف الكلاس أو إضافة ميزة جديدة له هنا):


### 3. تجربة التقطيع على وثيقة متعددة الموضوعات
**الفكرة ببساطة:**
سنجهز وثيقة بها كلام عن إطار LangChain وبنفس الوقت كلام عن برج إيفل وباريس.
سنرى كيف يقوم كلاس `ThresholdSemanticChunker` بفصلهم لقطع مستقلة تماماً.

In [ ]:
# sample multi-topic text
raw_text = """LangChain is a framework for building applications with LLMs.
LangChain provides modular abstraction to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination."""

source_doc = Document(
    page_content=raw_text,
    metadata={"source": "lecture_sample", "category": "AI & Geography"}
)

chunker = ThresholdSemanticChunker(threshold=0.7)
doc_chunks = chunker.split_documents([source_doc])

print(f"📦 Total chunks produced: {len(doc_chunks)}\n")
for idx, doc in enumerate(doc_chunks, 1):
    print(f"--- Chunk [{idx}] | Metadata: {doc.metadata} ---")
    print(f"{doc.page_content}\n")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب إنشاء وثيقة خاصة بك وتمريرها للـ chunker هنا):


### 4. الفهرسة في مخزن المتجهات (`FAISS`) وإنشاء المسترجع
**الفكرة ببساطة:**
بنحول القطع الدلالية الناتجة إلى متجهات ونخزنها في `FAISS`، ثم نحول المخزن إلى `Retriever` يجلب أفضل قطعتين (`k=2`).

In [ ]:
# index chunks into FAISS vector store
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(doc_chunks, embeddings)

retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# test retriever
query = "What tools can LangChain combine?"
retrieved = retriever.invoke(query)
print(f"🔍 Top retrieved chunk for: '{query}':\n")
print(retrieved[0].page_content)

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب الاستعلام عن موضوع مختلف مثل باريس وافحص السياق المسترجع هنا):


### 5. بناء مسار RAG متكامل بتعبيرات `LCEL`
**الفكرة ببساطة:**
بنربط السلسلة بالكامل:
`retriever` ⬅️ يغذي `ChatPromptTemplate` ⬅️ يمر للـ `ChatGroq` ⬅️ يخرج كنص صافي عبر `StrOutputParser`.

In [ ]:
# construct LCEL RAG chain
prompt = ChatPromptTemplate.from_template("""Answer the question based only on the following context:
Context:
{context}

Question:
{question}

Answer:""")

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.4)

rag_chain = (
    RunnableParallel(
        context=(lambda x: x["question"]) | retriever,
        question=lambda x: x["question"]
    )
    | prompt
    | llm
    | StrOutputParser()
)

# test with lecture question
q = "What is LangChain used for?"
answer = rag_chain.invoke({"question": q})

print(f"❓ Question: {q}\n")
print(f"💡 RAG Answer:\n{answer}")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب سؤال المسار عن أين يقع برج إيفل وشاهد الإجابة الدقيقة هنا):
